<a href="https://colab.research.google.com/github/Keerthivardhan08/heart_beat_rhythm/blob/main/Copy_of_CardioVision3D_ML_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import files

uploaded = files.upload()


Saving extention of Z-Alizadeh sani dataset.xlsx to extention of Z-Alizadeh sani dataset.xlsx


In [ ]:

from pathlib import Path

excel_files = list(Path("/content").glob("*.xlsx"))

print("Excel files found:", len(excel_files))

for file in excel_files:
    print("Filename:", file.name)
    print("Size:", round(file.stat().st_size / 1024, 2), "KB")


Excel files found: 1
Filename: extention of Z-Alizadeh sani dataset.xlsx
Size: 128.06 KB


In [ ]:

import pandas as pd

if not excel_files:
    raise FileNotFoundError("Upload the dataset first.")

if len(excel_files) != 1:
    raise ValueError(
        "Multiple Excel files found. Select the intended dataset first."
    )

file_path = excel_files[0]

# Inspect available worksheets.
workbook = pd.ExcelFile(file_path)
print("Worksheets:", workbook.sheet_names)

# Load the first worksheet.
df = pd.read_excel(file_path, sheet_name=0)

print("\nDataset dimensions:", df.shape)
print("Number of patients/records:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
for column in df.columns:
    print(repr(column))


Worksheets: ['Sheet 1 - Table 1', 'Sheet1']

Dataset dimensions: (303, 59)
Number of patients/records: 303
Number of columns: 59

Column names:
'Age'
'Weight'
'Length'
'Sex'
'BMI'
'DM'
'HTN'
'Current Smoker'
'EX-Smoker'
'FH'
'Obesity'
'CRF'
'CVA'
'Airway disease'
'Thyroid Disease'
'CHF'
'DLP'
'BP'
'PR'
'Edema'
'Weak Peripheral Pulse'
'Lung rales'
'Systolic Murmur'
'Diastolic Murmur'
'Typical Chest Pain'
'Dyspnea'
'Function Class'
'Atypical'
'Nonanginal'
'Exertional CP'
'LowTH Ang'
'Q Wave'
'St Elevation'
'St Depression'
'Tinversion'
'LVH'
'Poor R Progression'
'BBB'
'FBS'
'CR'
'TG'
'LDL'
'HDL'
'BUN'
'ESR'
'HB'
'K'
'Na'
'WBC'
'Lymph'
'Neut'
'PLT'
'EF-TTE'
'Region RWMA'
'VHD'
'LAD'
'LCX'
'RCA'
'Cath'


In [ ]:

target_candidates = ["LAD", "LCX", "RCA", "Cath"]

for column in target_candidates:
    print(f"\n{'=' * 40}")
    print(f"Column: {column}")
    print("Data type:", df[column].dtype)
    print("Missing values:", df[column].isna().sum())
    print("Unique values and counts:")
    print(df[column].value_counts(dropna=False).to_string())



Column: LAD
Data type: object
Missing values: 0
Unique values and counts:
LAD
Stenotic    177
Normal      126

Column: LCX
Data type: object
Missing values: 0
Unique values and counts:
LCX
Normal      184
Stenotic    119

Column: RCA
Data type: object
Missing values: 0
Unique values and counts:
RCA
Normal      189
Stenotic    114

Column: Cath
Data type: object
Missing values: 0
Unique values and counts:
Cath
CAD       216
Normal     87


In [ ]:

missing = df.isna().sum()

missing_report = (
    missing[missing > 0]
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

missing_report["missing_percent"] = (
    missing_report["missing_count"] / len(df) * 100
).round(2)

print("Columns with missing values:", len(missing_report))
display(missing_report)

print("\nDuplicate rows:", df.duplicated().sum())


Columns with missing values: 0


,missing_count,missing_percent



Duplicate rows: 0


In [ ]:

target_candidates = ["LAD", "LCX", "RCA", "Cath"]

for column in target_candidates:
    print(f"\n--- {column} ---")
    print("Data type:", df[column].dtype)
    print("Missing values:", df[column].isna().sum())
    print(df[column].value_counts(dropna=False).to_string())



--- LAD ---
Data type: object
Missing values: 0
LAD
Stenotic    177
Normal      126

--- LCX ---
Data type: object
Missing values: 0
LCX
Normal      184
Stenotic    119

--- RCA ---
Data type: object
Missing values: 0
RCA
Normal      189
Stenotic    114

--- Cath ---
Data type: object
Missing values: 0
Cath
CAD       216
Normal     87


In [ ]:

# Target columns supplied by the challenge.
target_columns = ["LAD", "LCX", "RCA", "Cath"]

# Explicitly forbidden as model inputs.
forbidden_features = {"LAD", "LCX", "RCA", "Cath"}

# These columns are candidates for review, not automatically approved.
candidate_features = [
    column
    for column in df.columns
    if column not in forbidden_features
]

print("Total columns:", len(df.columns))
print("Target/forbidden columns:", len(forbidden_features))
print("Remaining candidate features:", len(candidate_features))

print("\nCandidate feature names:")
for feature in candidate_features:
    print("-", feature)

# Safety check: no target is included as an input.
assert not forbidden_features.intersection(candidate_features)
print("\nLeakage exclusion check passed.")


Total columns: 59
Target/forbidden columns: 4
Remaining candidate features: 55

Candidate feature names:
- Age
- Weight
- Length
- Sex
- BMI
- DM
- HTN
- Current Smoker
- EX-Smoker
- FH
- Obesity
- CRF
- CVA
- Airway disease
- Thyroid Disease
- CHF
- DLP
- BP
- PR
- Edema
- Weak Peripheral Pulse
- Lung rales
- Systolic Murmur
- Diastolic Murmur
- Typical Chest Pain
- Dyspnea
- Function Class
- Atypical
- Nonanginal
- Exertional CP
- LowTH Ang
- Q Wave
- St Elevation
- St Depression
- Tinversion
- LVH
- Poor R Progression
- BBB
- FBS
- CR
- TG
- LDL
- HDL
- BUN
- ESR
- HB
- K
- Na
- WBC
- Lymph
- Neut
- PLT
- EF-TTE
- Region RWMA
- VHD

Leakage exclusion check passed.


In [ ]:

# Step 2A: Inspect the data types
print("DATA TYPES")
print(df.dtypes.to_string())

# Step 2B: Inspect categorical columns and their values
categorical_columns = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nCategorical columns:", len(categorical_columns))

for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False).to_dict())


DATA TYPES
Age                        int64
Weight                     int64
Length                     int64
Sex                       object
BMI                      float64
DM                         int64
HTN                        int64
Current Smoker             int64
EX-Smoker                  int64
FH                         int64
Obesity                   object
CRF                       object
CVA                       object
Airway disease            object
Thyroid Disease           object
CHF                       object
DLP                       object
BP                         int64
PR                         int64
Edema                      int64
Weak Peripheral Pulse     object
Lung rales                object
Systolic Murmur           object
Diastolic Murmur          object
Typical Chest Pain         int64
Dyspnea                   object
Function Class             int64
Atypical                  object
Nonanginal                object
Exertional CP             object

In [ ]:

# Step 3A: Find features with only one distinct value
constant_features = [
    col for col in candidate_features
    if df[col].nunique(dropna=False) <= 1
]

print("Constant features:", constant_features)

# Step 3B: Find features with very rare categories
# This is an audit only; we are not removing these automatically.
print("\nFeatures with categories appearing 10 times or fewer:")

for col in candidate_features:
    counts = df[col].value_counts(dropna=False)
    rare = counts[counts <= 10]

    if not rare.empty:
        print(f"{col}: {rare.to_dict()}")


Constant features: ['Exertional CP']

Features with categories appearing 10 times or fewer:
Age: {53: 10, 47: 9, 49: 9, 64: 9, 52: 9, 51: 8, 57: 8, 63: 7, 76: 7, 67: 7, 59: 7, 72: 6, 54: 6, 58: 6, 68: 6, 46: 5, 73: 5, 75: 5, 71: 5, 44: 4, 80: 4, 42: 4, 43: 4, 41: 3, 74: 3, 69: 3, 81: 3, 61: 3, 30: 2, 45: 2, 77: 2, 38: 2, 79: 2, 40: 1, 86: 1, 36: 1, 85: 1}
Weight: {63: 10, 65: 10, 73: 10, 66: 9, 82: 9, 72: 8, 76: 8, 67: 8, 78: 8, 68: 7, 90: 7, 89: 7, 81: 7, 84: 7, 79: 6, 61: 6, 88: 6, 85: 6, 83: 5, 71: 5, 54: 5, 62: 5, 57: 4, 55: 4, 53: 3, 92: 3, 87: 3, 69: 3, 98: 3, 58: 3, 50: 3, 59: 2, 95: 2, 100: 2, 56: 2, 77: 2, 91: 2, 97: 2, 86: 2, 48: 2, 108: 2, 51: 1, 120: 1, 93: 1, 103: 1, 102: 1, 94: 1, 105: 1}
Length: {155: 10, 153: 10, 169: 9, 164: 9, 163: 8, 152: 8, 172: 8, 174: 8, 161: 8, 162: 8, 166: 8, 178: 7, 167: 7, 171: 7, 177: 6, 157: 5, 158: 5, 156: 5, 159: 5, 176: 5, 173: 4, 148: 4, 179: 3, 180: 3, 181: 3, 185: 3, 151: 2, 147: 2, 149: 2, 145: 2, 186: 1, 183: 1, 144: 1, 188: 1, 187: 

In [ ]:

# Step 4A: Check whether Length is plausibly height in centimetres
# BMI formula, if Weight is kg and Length is cm:
# BMI = Weight / (Length in metres ** 2)

bmi_calculated = df["Weight"] / ((df["Length"] / 100) ** 2)
bmi_difference = (df["BMI"] - bmi_calculated).abs()

print("BMI consistency check")
print("Median absolute difference:", round(bmi_difference.median(), 4))
print("Maximum absolute difference:", round(bmi_difference.max(), 4))
print("Within 0.2 BMI units:", int((bmi_difference <= 0.2).sum()), "of", len(df))

print("\nPotentially important timing/definition checks:")
for col in [
    "Q Wave", "St Elevation", "St Depression", "Tinversion",
    "LVH", "BBB", "EF-TTE", "Region RWMA", "VHD"
]:
    print(f"{col}: {df[col].nunique(dropna=False)} distinct values")


BMI consistency check
Median absolute difference: 0.0
Maximum absolute difference: 0.0
Within 0.2 BMI units: 303 of 303

Potentially important timing/definition checks:
Q Wave: 2 distinct values
St Elevation: 2 distinct values
St Depression: 2 distinct values
Tinversion: 2 distinct values
LVH: 2 distinct values
BBB: 3 distinct values
EF-TTE: 11 distinct values
Region RWMA: 5 distinct values
VHD: 4 distinct values


In [ ]:

# Step 5A: Define target columns and mandatory exclusions
target_columns = ["Cath", "LAD", "LCX", "RCA"]

constant_features = ["Exertional CP"]
redundant_features = ["Weight", "Length"]

# These are provisional pending confirmation of measurement timing.
timing_review_features = [
    "Q Wave",
    "St Elevation",
    "St Depression",
    "Tinversion",
    "LVH",
    "Poor R Progression",
    "BBB",
    "EF-TTE",
    "Region RWMA",
    "VHD",
]

# Initial candidate list for further review.
excluded_for_baseline = (
    set(target_columns)
    | set(constant_features)
    | set(redundant_features)
)

provisional_features = [
    col for col in df.columns
    if col not in excluded_for_baseline
]

print("Provisional feature count:", len(provisional_features))
print("Features awaiting timing review:")
print([col for col in timing_review_features
       if col in provisional_features])

# Safety checks
assert not set(target_columns).intersection(provisional_features)
assert "Exertional CP" not in provisional_features
assert "Weight" not in provisional_features
assert "Length" not in provisional_features

print("\nLeakage exclusion checks passed.")
print("Original dataframe unchanged:", df.shape == (303, 59))


Provisional feature count: 52
Features awaiting timing review:
['Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'LVH', 'Poor R Progression', 'BBB', 'EF-TTE', 'Region RWMA', 'VHD']

Leakage exclusion checks passed.
Original dataframe unchanged: True


In [ ]:

# Step 6A: Compare Cath with the artery-based CAD definition

artery_columns = ["LAD", "LCX", "RCA"]

# CAD is implied if at least one artery is labelled Stenotic.
cad_from_arteries = df[artery_columns].eq("Stenotic").any(axis=1)

# Cath is the overall CAD label.
cad_from_cath = df["Cath"].eq("CAD")

# Count disagreements between the two definitions.
mismatch_mask = cad_from_arteries != cad_from_cath

print("Rows checked:", len(df))
print("CAD according to Cath:", int(cad_from_cath.sum()))
print("CAD implied by artery labels:", int(cad_from_arteries.sum()))
print("Disagreements:", int(mismatch_mask.sum()))

if mismatch_mask.any():
    print("\nRows with disagreements:")
    print(df.loc[mismatch_mask, ["Cath", "LAD", "LCX", "RCA"]])
else:
    print("\nAll four target labels are consistent.")


Rows checked: 303
CAD according to Cath: 216
CAD implied by artery labels: 217
Disagreements: 1

Rows with disagreements:
      Cath       LAD     LCX     RCA
93  Normal  Stenotic  Normal  Normal


In [ ]:
# Step 7A: Inspect the disputed record without changing the dataset

row_index = 93

print("Disputed record:")
display(df.loc[[row_index]].T)

print("\nTarget label combinations:")
print(
    df.groupby(["Cath", "LAD", "LCX", "RCA"], dropna=False)
      .size()
      .sort_values(ascending=False)
      .rename("count")
      .to_frame()
)

Disputed record:


,93
Age,65
Weight,73
Length,165
Sex,Male
BMI,26.81359
DM,0
HTN,1
Current Smoker,1
EX-Smoker,0
FH,0



Target label combinations:
                                   count
Cath   LAD      LCX      RCA            
Normal Normal   Normal   Normal       86
CAD    Stenotic Stenotic Stenotic     63
                Normal   Normal       56
                Stenotic Normal       33
                Normal   Stenotic     24
       Normal   Normal   Stenotic     17
                Stenotic Normal       13
                         Stenotic     10
Normal Stenotic Normal   Normal        1


In [ ]:

# Step 8A: Define leakage-safe provisional features

target_columns = ["Cath", "LAD", "LCX", "RCA"]
constant_features = ["Exertional CP"]
redundant_features = ["Weight", "Length"]

excluded_features = set(target_columns + constant_features + redundant_features)

feature_columns = [
    col for col in df.columns
    if col not in excluded_features
]

# Confirm no target columns appear in the predictors.
assert not set(feature_columns).intersection(target_columns)
assert "Exertional CP" not in feature_columns
assert "Weight" not in feature_columns
assert "Length" not in feature_columns

print("Provisional feature count:", len(feature_columns))
print("Excluded target columns:", target_columns)
print("Excluded constant features:", constant_features)
print("Excluded redundant features:", redundant_features)
print("\nProvisional features:")
print(feature_columns)
print("\nOriginal dataframe shape:", df.shape)


Provisional feature count: 52
Excluded target columns: ['Cath', 'LAD', 'LCX', 'RCA']
Excluded constant features: ['Exertional CP']
Excluded redundant features: ['Weight', 'Length']

Provisional features:
['Age', 'Sex', 'BMI', 'DM', 'HTN', 'Current Smoker', 'EX-Smoker', 'FH', 'Obesity', 'CRF', 'CVA', 'Airway disease', 'Thyroid Disease', 'CHF', 'DLP', 'BP', 'PR', 'Edema', 'Weak Peripheral Pulse', 'Lung rales', 'Systolic Murmur', 'Diastolic Murmur', 'Typical Chest Pain', 'Dyspnea', 'Function Class', 'Atypical', 'Nonanginal', 'LowTH Ang', 'Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'LVH', 'Poor R Progression', 'BBB', 'FBS', 'CR', 'TG', 'LDL', 'HDL', 'BUN', 'ESR', 'HB', 'K', 'Na', 'WBC', 'Lymph', 'Neut', 'PLT', 'EF-TTE', 'Region RWMA', 'VHD']

Original dataframe shape: (303, 59)


In [ ]:

# Step 9A: Reproducible split for the overall CAD task

from sklearn.model_selection import train_test_split

TARGET = "Cath"
RANDOM_STATE = 42
TEST_SIZE = 0.20

X = df[feature_columns].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts())
print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting class distribution:")
print(y_test.value_counts())
print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True).round(3))

# Leakage and split-integrity checks
assert len(X_train) + len(X_test) == len(df)
assert X_train.index.intersection(X_test.index).empty
assert not set(target_columns).intersection(X_train.columns)
assert X_train.columns.equals(X_test.columns)

print("\nSplit-integrity checks passed.")


Training rows: 242
Testing rows: 61

Training class distribution:
Cath
CAD       173
Normal     69
Name: count, dtype: int64

Training class proportions:
Cath
CAD       0.715
Normal    0.285
Name: proportion, dtype: float64

Testing class distribution:
Cath
CAD       43
Normal    18
Name: count, dtype: int64

Testing class proportions:
Cath
CAD       0.705
Normal    0.295
Name: proportion, dtype: float64

Split-integrity checks passed.


In [ ]:

# Step 10A: Inspect predictor data types

print("Pandas data types:")
print(X_train.dtypes.value_counts())

numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical features:", len(numeric_features))
print(numeric_features)

print("\nCategorical features:", len(categorical_features))
print(categorical_features)

# Every predictor must be assigned to exactly one group.
assert set(numeric_features).isdisjoint(categorical_features)
assert set(numeric_features) | set(categorical_features) == set(X_train.columns)

print("\nFeature-type checks passed.")


Pandas data types:
int64      27
object     20
float64     5
Name: count, dtype: int64

Numerical features: 32
['Age', 'BMI', 'DM', 'HTN', 'Current Smoker', 'EX-Smoker', 'FH', 'BP', 'PR', 'Edema', 'Typical Chest Pain', 'Function Class', 'Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'FBS', 'CR', 'TG', 'LDL', 'HDL', 'BUN', 'ESR', 'HB', 'K', 'Na', 'WBC', 'Lymph', 'Neut', 'PLT', 'EF-TTE', 'Region RWMA']

Categorical features: 20
['Sex', 'Obesity', 'CRF', 'CVA', 'Airway disease', 'Thyroid Disease', 'CHF', 'DLP', 'Weak Peripheral Pulse', 'Lung rales', 'Systolic Murmur', 'Diastolic Murmur', 'Dyspnea', 'Atypical', 'Nonanginal', 'LowTH Ang', 'LVH', 'Poor R Progression', 'BBB', 'VHD']

Feature-type checks passed.


In [ ]:
print("Dataset loaded:", "df" in globals())
print("Training data ready:", "X_train" in globals())
print("Training labels ready:", "y_train" in globals())
print("Baseline pipeline ready:", "baseline_model" in globals())

Dataset loaded: True
Training data ready: True
Training labels ready: True
Baseline pipeline ready: False


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Process numerical features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Process categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Build the complete baseline model
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

print("Baseline pipeline defined successfully.")
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total input features:", len(feature_columns))
print("Training records:", len(X_train))
print("Held-out test records:", len(X_test))

Baseline pipeline defined successfully.
Numerical features: 32
Categorical features: 20
Total input features: 52
Training records: 242
Held-out test records: 61


In [ ]:
# Train the Logistic Regression baseline
baseline_model.fit(X_train, y_train)

print("Baseline training complete!")
print("Training records:", len(X_train))
print("Held-out test records:", len(X_test))
print("Classes learned:", baseline_model.classes_)

Baseline training complete!
Training records: 242
Held-out test records: 61
Classes learned: ['CAD' 'Normal']


In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.dummy import DummyClassifier

# 1. Predict on the untouched test set
y_pred = baseline_model.predict(X_test)
y_proba = baseline_model.predict_proba(X_test)

# 2. Identify CAD as the positive class
cad_index = list(baseline_model.classes_).index("CAD")
y_test_binary = (y_test == "CAD").astype(int)
cad_probabilities = y_proba[:, cad_index]

# 3. Calculate evaluation metrics
metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "CAD Precision": precision_score(
        y_test, y_pred, pos_label="CAD", zero_division=0
    ),
    "CAD Recall": recall_score(
        y_test, y_pred, pos_label="CAD", zero_division=0
    ),
    "CAD F1-score": f1_score(
        y_test, y_pred, pos_label="CAD", zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test_binary, cad_probabilities),
}

print("=== LOGISTIC REGRESSION: HELD-OUT TEST RESULTS ===")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

# 4. Confusion matrix: rows = actual, columns = predicted
cm = confusion_matrix(
    y_test, y_pred, labels=["CAD", "Normal"]
)

print("\n=== CONFUSION MATRIX ===")
print("Labels order: CAD, Normal")
print(cm)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(
    y_test, y_pred,
    labels=["CAD", "Normal"],
    zero_division=0,
))

# 5. Compare against a majority-class baseline
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)

print("=== MAJORITY-CLASS BASELINE ===")
print(f"Accuracy: {accuracy_score(y_test, dummy_pred):.4f}")
print(f"CAD F1-score: {f1_score(y_test, dummy_pred, pos_label='CAD', zero_division=0):.4f}")

# 6. Check class counts in the test set
print("\n=== TEST SET CLASS COUNTS ===")
print(y_test.value_counts())


=== LOGISTIC REGRESSION: HELD-OUT TEST RESULTS ===
Accuracy: 0.8197
CAD Precision: 0.8333
CAD Recall: 0.9302
CAD F1-score: 0.8791
ROC-AUC: 0.8643

=== CONFUSION MATRIX ===
Labels order: CAD, Normal
[[40  3]
 [ 8 10]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

         CAD       0.83      0.93      0.88        43
      Normal       0.77      0.56      0.65        18

    accuracy                           0.82        61
   macro avg       0.80      0.74      0.76        61
weighted avg       0.81      0.82      0.81        61

=== MAJORITY-CLASS BASELINE ===
Accuracy: 0.7049
CAD F1-score: 0.8269

=== TEST SET CLASS COUNTS ===
Cath
CAD       43
Normal    18
Name: count, dtype: int64


In [ ]:

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    make_scorer,
    recall_score,
    roc_auc_score,
)
import numpy as np
import pandas as pd

# Keep each fold's CAD/Normal proportions approximately balanced
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Explicitly calculate ROC-AUC with CAD as the positive class
def cad_roc_auc(estimator, X_val, y_val):
    cad_index = list(estimator.classes_).index("CAD")
    cad_prob = estimator.predict_proba(X_val)[:, cad_index]
    y_binary = (y_val == "CAD").astype(int)
    return roc_auc_score(y_binary, cad_prob)

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "cad_recall": make_scorer(
        recall_score, pos_label="CAD", zero_division=0
    ),
    "normal_recall": make_scorer(
        recall_score, pos_label="Normal", zero_division=0
    ),
    "cad_roc_auc": cad_roc_auc,
}

cv_results = cross_validate(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

# Show each fold and its mean/std
metric_names = list(scoring.keys())
summary = []

for metric in metric_names:
    scores = cv_results[f"test_{metric}"]
    summary.append({
        "Metric": metric,
        "Mean": scores.mean(),
        "Std": scores.std(),
    })

cv_summary = pd.DataFrame(summary)
print("=== 5-FOLD CROSS-VALIDATION: TRAINING DATA ONLY ===")
print(cv_summary.round(4).to_string(index=False))

print("\nFold-level CAD ROC-AUC:")
print(np.round(cv_results["test_cad_roc_auc"], 4))


=== 5-FOLD CROSS-VALIDATION: TRAINING DATA ONLY ===
           Metric   Mean    Std
         accuracy 0.8764 0.0588
balanced_accuracy 0.8485 0.0617
         macro_f1 0.8502 0.0688
       cad_recall 0.9134 0.0570
    normal_recall 0.7835 0.0763
      cad_roc_auc 0.9377 0.0350

Fold-level CAD ROC-AUC:
[0.9082 0.8918 0.9912 0.9433 0.9538]


In [ ]:

from sklearn.base import clone
from sklearn.model_selection import cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import pandas as pd

# Features requiring prediction-time availability review
timing_review_features = [
    "Q Wave", "St Elevation", "St Depression", "Tinversion",
    "LVH", "Poor R Progression", "BBB", "EF-TTE",
    "Region RWMA", "VHD"
]

# Sensitivity-analysis feature set
reduced_features = [
    col for col in feature_columns
    if col not in timing_review_features
]

# Infer data types from the training set only
reduced_numeric = X_train[reduced_features].select_dtypes(
    include=["number"]
).columns.tolist()

reduced_categorical = X_train[reduced_features].select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Build a fresh pipeline for the reduced feature set
reduced_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), reduced_numeric),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), reduced_categorical)
])

reduced_model = Pipeline([
    ("preprocessor", reduced_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000, random_state=42
    ))
])

# Use exactly the same training folds as before
reduced_cv_results = cross_validate(
    reduced_model,
    X_train[reduced_features],
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

# Compare the cross-validation means
comparison = pd.DataFrame({
    "Metric": metric_names,
    "Current 52-feature model": [
        cv_results[f"test_{m}"].mean() for m in metric_names
    ],
    "Reduced feature model": [
        reduced_cv_results[f"test_{m}"].mean() for m in metric_names
    ],
})

comparison["Difference"] = (
    comparison["Reduced feature model"]
    - comparison["Current 52-feature model"]
)

print("=== FEATURE-SENSITIVITY ANALYSIS ===")
print("Current feature count:", len(feature_columns))
print("Reduced feature count:", len(reduced_features))
print("Features removed:", timing_review_features)
print(comparison.round(4).to_string(index=False))


=== FEATURE-SENSITIVITY ANALYSIS ===
Current feature count: 52
Reduced feature count: 42
Features removed: ['Q Wave', 'St Elevation', 'St Depression', 'Tinversion', 'LVH', 'Poor R Progression', 'BBB', 'EF-TTE', 'Region RWMA', 'VHD']
           Metric  Current 52-feature model  Reduced feature model  Difference
         accuracy                    0.8764                 0.8598     -0.0167
balanced_accuracy                    0.8485                 0.8155     -0.0329
         macro_f1                    0.8502                 0.8249     -0.0253
       cad_recall                    0.9134                 0.9190      0.0055
    normal_recall                    0.7835                 0.7121     -0.0714
      cad_roc_auc                    0.9377                 0.9139     -0.0237


In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.base import clone
import pandas as pd

# Random Forest baseline
random_forest_model = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# Evaluate only on training folds
rf_cv_results = cross_validate(
    random_forest_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=False
)

# Compare with the existing Logistic Regression CV results
comparison_rows = []

for metric in metric_names:
    log_scores = cv_results[f"test_{metric}"]
    rf_scores = rf_cv_results[f"test_{metric}"]

    comparison_rows.append({
        "Metric": metric,
        "Logistic Regression mean": log_scores.mean(),
        "Logistic Regression std": log_scores.std(),
        "Random Forest mean": rf_scores.mean(),
        "Random Forest std": rf_scores.std(),
        "RF minus Logistic": rf_scores.mean() - log_scores.mean()
    })

algorithm_comparison = pd.DataFrame(comparison_rows)

print("=== ALGORITHM COMPARISON: TRAINING DATA ONLY ===")
print(algorithm_comparison.round(4).to_string(index=False))



=== ALGORITHM COMPARISON: TRAINING DATA ONLY ===
           Metric  Logistic Regression mean  Logistic Regression std  Random Forest mean  Random Forest std  RF minus Logistic
         accuracy                    0.8764                   0.0588              0.8639             0.0371            -0.0125
balanced_accuracy                    0.8485                   0.0617              0.8319             0.0591            -0.0166
         macro_f1                    0.8502                   0.0688              0.8313             0.0486            -0.0189
       cad_recall                    0.9134                   0.0570              0.9077             0.0207            -0.0057
    normal_recall                    0.7835                   0.0763              0.7560             0.1144            -0.0275
      cad_roc_auc                    0.9377                   0.0350              0.9245             0.0340            -0.0132


In [ ]:

from sklearn.model_selection import train_test_split
import pandas as pd

TARGETS = ["Cath", "LAD", "LCX", "RCA"]
FORBIDDEN_FEATURES = set(TARGETS)

# Verify no target label is used as an input feature
assert not (set(feature_columns) & FORBIDDEN_FEATURES), (
    "Leakage detected: a target column is in the feature list."
)

# Use the same feature columns for every target
X_all = df[feature_columns].copy()

# Preserve the existing CAD split; create separate splits for vessels
target_splits = {
    "Cath": (X_train.copy(), X_test.copy(),
             y_train.copy(), y_test.copy())
}

for target in ["LAD", "LCX", "RCA"]:
    y_target = df[target].copy()

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all,
        y_target,
        test_size=0.20,
        random_state=42,
        stratify=y_target
    )

    target_splits[target] = (X_tr, X_te, y_tr, y_te)

# Print split sizes and label distribution
for target, (X_tr, X_te, y_tr, y_te) in target_splits.items():
    print(f"\n=== {target} ===")
    print("Train records:", len(X_tr))
    print("Test records:", len(X_te))
    print("Training labels:")
    print(y_tr.value_counts().to_dict())
    print("Test labels:")
    print(y_te.value_counts().to_dict())

print("\nAll four targets prepared.")
print("Input feature count:", len(feature_columns))
print("Target columns excluded:", sorted(FORBIDDEN_FEATURES))



=== Cath ===
Train records: 242
Test records: 61
Training labels:
{'CAD': 173, 'Normal': 69}
Test labels:
{'CAD': 43, 'Normal': 18}

=== LAD ===
Train records: 242
Test records: 61
Training labels:
{'Stenotic': 141, 'Normal': 101}
Test labels:
{'Stenotic': 36, 'Normal': 25}

=== LCX ===
Train records: 242
Test records: 61
Training labels:
{'Normal': 147, 'Stenotic': 95}
Test labels:
{'Normal': 37, 'Stenotic': 24}

=== RCA ===
Train records: 242
Test records: 61
Training labels:
{'Normal': 151, 'Stenotic': 91}
Test labels:
{'Normal': 38, 'Stenotic': 23}

All four targets prepared.
Input feature count: 52
Target columns excluded: ['Cath', 'LAD', 'LCX', 'RCA']


In [ ]:

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

target_models = {}
target_metrics = []

for target, (X_tr, X_te, y_tr, y_te) in target_splits.items():

    # Determine feature types from this target's training data
    num_cols = X_tr.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = X_tr.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    # Build preprocessing that learns only from training records
    prep = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    model = Pipeline([
        ("preprocessor", prep),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ])

    # Train and predict
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)

    # Define the positive class for each task
    positive_label = "CAD" if target == "Cath" else "Stenotic"
    positive_index = list(model.classes_).index(positive_label)

    y_true_binary = (y_te == positive_label).astype(int)
    positive_prob = y_prob[:, positive_index]

    # Compute held-out metrics
    accuracy = accuracy_score(y_te, y_pred)
    balanced_acc = balanced_accuracy_score(y_te, y_pred)
    precision = precision_score(
        y_te, y_pred, pos_label=positive_label, zero_division=0
    )
    recall = recall_score(
        y_te, y_pred, pos_label=positive_label, zero_division=0
    )
    f1 = f1_score(
        y_te, y_pred, pos_label=positive_label, zero_division=0
    )
    auc = roc_auc_score(y_true_binary, positive_prob)

    target_models[target] = model
    target_metrics.append({
        "Target": target,
        "Accuracy": accuracy,
        "Balanced accuracy": balanced_acc,
        f"{positive_label} precision": precision,
        f"{positive_label} recall": recall,
        f"{positive_label} F1": f1,
        "ROC-AUC": auc
    })

    print(f"\n{'=' * 55}")
    print(f"TARGET: {target} | Positive class: {positive_label}")
    print("Classes:", list(model.classes_))
    print("Confusion matrix (actual rows, predicted columns):")
    print("Label order:", list(model.classes_))
    print(confusion_matrix(y_te, y_pred, labels=model.classes_))
    print("\nClassification report:")
    print(classification_report(
        y_te, y_pred, labels=model.classes_, zero_division=0
    ))

print("\n=== HELD-OUT RESULTS: ALL FOUR TARGETS ===")
results_df = pd.DataFrame(target_metrics)
print(results_df.round(4).to_string(index=False))



TARGET: Cath | Positive class: CAD
Classes: ['CAD', 'Normal']
Confusion matrix (actual rows, predicted columns):
Label order: ['CAD', 'Normal']
[[40  3]
 [ 8 10]]

Classification report:
              precision    recall  f1-score   support

         CAD       0.83      0.93      0.88        43
      Normal       0.77      0.56      0.65        18

    accuracy                           0.82        61
   macro avg       0.80      0.74      0.76        61
weighted avg       0.81      0.82      0.81        61


TARGET: LAD | Positive class: Stenotic
Classes: ['Normal', 'Stenotic']
Confusion matrix (actual rows, predicted columns):
Label order: ['Normal', 'Stenotic']
[[17  8]
 [ 8 28]]

Classification report:
              precision    recall  f1-score   support

      Normal       0.68      0.68      0.68        25
    Stenotic       0.78      0.78      0.78        36

    accuracy                           0.74        61
   macro avg       0.73      0.73      0.73        61
weighted av

In [ ]:

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    make_scorer, recall_score, precision_score,
    f1_score, roc_auc_score
)
import pandas as pd
import numpy as np

def make_positive_auc_scorer(positive_label):
    def scorer(estimator, X_val, y_val):
        classes = list(estimator.classes_)
        positive_idx = classes.index(positive_label)
        probabilities = estimator.predict_proba(X_val)[:, positive_idx]
        y_binary = (y_val == positive_label).astype(int)
        return roc_auc_score(y_binary, probabilities)
    return scorer

cv_summary_rows = []
cv_fold_results = {}

for target, (X_tr, X_te, y_tr, y_te) in target_splits.items():
    positive_label = "CAD" if target == "Cath" else "Stenotic"

    target_scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "precision": make_scorer(
            precision_score,
            pos_label=positive_label,
            zero_division=0
        ),
        "recall": make_scorer(
            recall_score,
            pos_label=positive_label,
            zero_division=0
        ),
        "macro_f1": "f1_macro",
        "roc_auc": make_positive_auc_scorer(positive_label)
    }

    # Fresh, reproducible stratified folds for this target
    target_cv = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=42
    )

    results = cross_validate(
        clone(target_models[target]),
        X_tr,
        y_tr,
        cv=target_cv,
        scoring=target_scoring,
        n_jobs=-1,
        return_train_score=False
    )

    cv_fold_results[target] = results

    row = {"Target": target}
    for metric in target_scoring:
        values = results[f"test_{metric}"]
        row[f"{metric} mean"] = values.mean()
        row[f"{metric} std"] = values.std()

    cv_summary_rows.append(row)

    print(f"\n{target} — positive class: {positive_label}")
    print("ROC-AUC by fold:",
          np.round(results["test_roc_auc"], 4))

cv_all_targets_df = pd.DataFrame(cv_summary_rows)

print("\n=== FIVE-FOLD CROSS-VALIDATION SUMMARY ===")
print(cv_all_targets_df.round(4).to_string(index=False))



Cath — positive class: CAD
ROC-AUC by fold: [0.9082 0.8918 0.9912 0.9433 0.9538]

LAD — positive class: Stenotic
ROC-AUC by fold: [0.8466 0.7704 0.7893 0.7286 0.8357]

LCX — positive class: Stenotic
ROC-AUC by fold: [0.7965 0.5895 0.6534 0.5681 0.6915]

RCA — positive class: Stenotic
ROC-AUC by fold: [0.8297 0.7965 0.587  0.7056 0.6741]

=== FIVE-FOLD CROSS-VALIDATION SUMMARY ===
Target  accuracy mean  accuracy std  balanced_accuracy mean  balanced_accuracy std  precision mean  precision std  recall mean  recall std  macro_f1 mean  macro_f1 std  roc_auc mean  roc_auc std
  Cath         0.8764        0.0588                  0.8485                 0.0617          0.9129         0.0328       0.9134      0.0570         0.8502        0.0688        0.9377       0.0350
   LAD         0.7149        0.0200                  0.7093                 0.0317          0.7717         0.0677       0.7443      0.0702         0.7052        0.0247        0.7941       0.0433
   LCX         0.6115        0.

In [ ]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
import pandas as pd

baseline_comparison = []

for target, (X_tr, X_te, y_tr, y_te) in target_splits.items():
    positive_label = "CAD" if target == "Cath" else "Stenotic"

    # Existing trained Logistic Regression model
    model = target_models[target]
    model_pred = model.predict(X_te)
    model_prob = model.predict_proba(X_te)[
        :, list(model.classes_).index(positive_label)
    ]

    # Simple majority-class baseline, fitted on training data only
    dummy = DummyClassifier(strategy="most_frequent")
    dummy.fit(X_tr, y_tr)
    dummy_pred = dummy.predict(X_te)

    y_binary = (y_te == positive_label).astype(int)

    baseline_comparison.append({
        "Target": target,
        "Model accuracy": accuracy_score(y_te, model_pred),
        "Majority accuracy": accuracy_score(y_te, dummy_pred),
        "Model balanced accuracy": balanced_accuracy_score(
            y_te, model_pred
        ),
        "Majority balanced accuracy": balanced_accuracy_score(
            y_te, dummy_pred
        ),
        "Model macro F1": f1_score(
            y_te, model_pred, average="macro", zero_division=0
        ),
        "Majority macro F1": f1_score(
            y_te, dummy_pred, average="macro", zero_division=0
        ),
        "Model ROC-AUC": roc_auc_score(y_binary, model_prob),
    })

baseline_comparison_df = pd.DataFrame(baseline_comparison)

print("=== MODEL VS MAJORITY BASELINE ===")
print(baseline_comparison_df.round(4).to_string(index=False))


=== MODEL VS MAJORITY BASELINE ===
Target  Model accuracy  Majority accuracy  Model balanced accuracy  Majority balanced accuracy  Model macro F1  Majority macro F1  Model ROC-AUC
  Cath          0.8197             0.7049                   0.7429                         0.5          0.7621             0.4135         0.8643
   LAD          0.7377             0.5902                   0.7289                         0.5          0.7289             0.3711         0.8267
   LCX          0.6066             0.6066                   0.5878                         0.5          0.5878             0.3776         0.6363
   RCA          0.6885             0.6230                   0.6470                         0.5          0.6510             0.3838         0.7426


In [ ]:
###

In [ ]:

# CardioVision3D — Feature audit (Step 1)

print("Dataset shape:", df.shape)
print("\nAll dataset columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nTarget columns:")
print([c for c in ["Cath", "LAD", "LCX", "RCA"] if c in df.columns])


Dataset shape: (303, 59)

All dataset columns:
01. Age
02. Weight
03. Length
04. Sex
05. BMI
06. DM
07. HTN
08. Current Smoker
09. EX-Smoker
10. FH
11. Obesity
12. CRF
13. CVA
14. Airway disease
15. Thyroid Disease
16. CHF
17. DLP
18. BP
19. PR
20. Edema
21. Weak Peripheral Pulse
22. Lung rales
23. Systolic Murmur
24. Diastolic Murmur
25. Typical Chest Pain
26. Dyspnea
27. Function Class
28. Atypical
29. Nonanginal
30. Exertional CP
31. LowTH Ang
32. Q Wave
33. St Elevation
34. St Depression
35. Tinversion
36. LVH
37. Poor R Progression
38. BBB
39. FBS
40. CR
41. TG
42. LDL
43. HDL
44. BUN
45. ESR
46. HB
47. K
48. Na
49. WBC
50. Lymph
51. Neut
52. PLT
53. EF-TTE
54. Region RWMA
55. VHD
56. LAD
57. LCX
58. RCA
59. Cath

Target columns:
['Cath', 'LAD', 'LCX', 'RCA']


In [ ]:

# CardioVision3D — Step 2: Define provisional feature groups

targets = ["Cath", "LAD", "LCX", "RCA"]

# Exclude the four prediction targets from all input features.
excluded = set(targets) | {"Exertional CP", "Weight", "Length"}

# Experiment A: information categories intended for the basic model.
basic_features = [
    "Age", "Sex", "BMI",
    "DM", "HTN", "Current Smoker", "EX-Smoker", "FH",
    "Obesity", "CRF", "CVA", "Airway disease",
    "Thyroid Disease", "CHF", "DLP",
    "BP", "PR", "Edema", "Weak Peripheral Pulse",
    "Lung rales", "Systolic Murmur", "Diastolic Murmur",
    "Typical Chest Pain", "Dyspnea", "Function Class",
    "Atypical", "Nonanginal", "LowTH Ang",
    "FBS", "CR", "TG", "LDL", "HDL", "BUN", "ESR",
    "HB", "K", "Na", "WBC", "Lymph", "Neut", "PLT"
]

# Additional measurements for Experiment B.
expanded_features = [
    "Q Wave", "St Elevation", "St Depression", "Tinversion",
    "LVH", "Poor R Progression", "BBB",
    "EF-TTE", "Region RWMA", "VHD"
]

# Apply exclusions and verify the actual dataset schema.
basic_features = [
    c for c in basic_features
    if c not in excluded
]
expanded_features = [
    c for c in expanded_features
    if c not in excluded
]

missing_basic = [c for c in basic_features if c not in df.columns]
missing_expanded = [c for c in expanded_features if c not in df.columns]

print("Experiment A features:", len(basic_features))
print("Experiment B additional features:", len(expanded_features))
print("Experiment B total features:", len(set(basic_features + expanded_features)))
print("Missing basic columns:", missing_basic)
print("Missing expanded columns:", missing_expanded)
print("Target columns excluded:", all(c not in basic_features + expanded_features for c in targets))

assert not missing_basic and not missing_expanded, (
    "A column name does not match the dataset. Check the printed missing columns."
)
assert not (set(basic_features + expanded_features) & set(targets))


Experiment A features: 42
Experiment B additional features: 10
Experiment B total features: 52
Missing basic columns: []
Missing expanded columns: []
Target columns excluded: True


In [ ]:

# CardioVision3D — Step 3: Feature-group comparison
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

feature_sets = {
    "A_Basic_42": basic_features,
    "B_Expanded_52": basic_features + expanded_features,
}

targets_to_test = ["Cath", "LAD", "LCX", "RCA"]

scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "macro_f1": "f1_macro",
    "roc_auc": "roc_auc",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for target in targets_to_test:
    X_train_target = target_splits[target][0]
    y_train_target = target_splits[target][2]

    for set_name, columns in feature_sets.items():
        X = X_train_target[columns].copy()

        numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
        categorical_cols = [
            c for c in X.columns if c not in numeric_cols
        ]

        preprocessor = ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_cols),
        ])

        model = Pipeline([
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(
                max_iter=2000, random_state=42
            )),
        ])

        scores = cross_validate(
            model, X, y_train_target,
            cv=cv, scoring=scoring,
            return_train_score=False,
            n_jobs=-1,
            error_score="raise",
        )

        results.append({
            "Target": target,
            "Feature set": set_name,
            "Features": len(columns),
            **{
                f"{metric} mean": scores[f"test_{metric}"].mean()
                for metric in scoring
            },
            **{
                f"{metric} std": scores[f"test_{metric}"].std()
                for metric in scoring
            },
        })

comparison_df = pd.DataFrame(results)
print(comparison_df.round(4).to_string(index=False))


Target   Feature set  Features  accuracy mean  balanced_accuracy mean  macro_f1 mean  roc_auc mean  accuracy std  balanced_accuracy std  macro_f1 std  roc_auc std
  Cath    A_Basic_42        42         0.8598                  0.8155         0.8249        0.9139        0.0540                 0.0579        0.0648       0.0436
  Cath B_Expanded_52        52         0.8764                  0.8485         0.8502        0.9377        0.0588                 0.0617        0.0688       0.0350
   LAD    A_Basic_42        42         0.6776                  0.6673         0.6668        0.7529        0.0600                 0.0652        0.0630       0.0447
   LAD B_Expanded_52        52         0.7149                  0.7093         0.7052        0.7941        0.0200                 0.0317        0.0247       0.0433
   LCX    A_Basic_42        42         0.6567                  0.6243         0.6269        0.6922        0.0518                 0.0476        0.0497       0.0869
   LCX B_Expanded_52  

In [ ]:

# CardioVision3D — Step 4: Fold-by-fold stability check

from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd

fold_rows = []

for target in ["Cath", "LAD", "LCX", "RCA"]:
    X_all, _, y_all, _ = target_splits[target]

    # Use identical folds for both feature sets.
    splits = list(cv.split(X_all[basic_features], y_all))

    for feature_name, columns in feature_sets.items():
        X = X_all[columns].copy()

        numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
        categorical_cols = [c for c in X.columns if c not in numeric_cols]

        prep = ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encode", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_cols),
        ])

        model = Pipeline([
            ("prep", prep),
            ("clf", LogisticRegression(max_iter=2000, random_state=42)),
        ])

        scores = cross_validate(
            model, X, y_all,
            cv=splits,
            scoring={
                "balanced_accuracy": "balanced_accuracy",
                "roc_auc": "roc_auc",
            },
            n_jobs=-1,
            error_score="raise",
        )

        for fold in range(5):
            fold_rows.append({
                "Target": target,
                "Feature set": feature_name,
                "Fold": fold + 1,
                "Balanced accuracy": scores["test_balanced_accuracy"][fold],
                "ROC-AUC": scores["test_roc_auc"][fold],
            })

fold_df = pd.DataFrame(fold_rows)

for target in ["Cath", "LAD", "LCX", "RCA"]:
    a = fold_df[
        (fold_df["Target"] == target) &
        (fold_df["Feature set"] == "A_Basic_42")
    ].sort_values("Fold")

    b = fold_df[
        (fold_df["Target"] == target) &
        (fold_df["Feature set"] == "B_Expanded_52")
    ].sort_values("Fold")

    print(f"\n=== {target}: B minus A, paired folds ===")
    print(pd.DataFrame({
        "Fold": a["Fold"].values,
        "Delta balanced accuracy": (
            b["Balanced accuracy"].values -
            a["Balanced accuracy"].values
        ),
        "Delta ROC-AUC": (
            b["ROC-AUC"].values - a["ROC-AUC"].values
        ),
    }).round(4).to_string(index=False))



=== Cath: B minus A, paired folds ===
 Fold  Delta balanced accuracy  Delta ROC-AUC
    1                   0.0857         0.0082
    2                  -0.0429         0.0347
    3                   0.0000         0.0022
    4                   0.0861         0.0462
    5                   0.0357         0.0273

=== LAD: B minus A, paired folds ===
 Fold  Delta balanced accuracy  Delta ROC-AUC
    1                   0.0422         0.0983
    2                   0.0179         0.0578
    3                   0.0964         0.0607
    4                   0.0750        -0.0071
    5                  -0.0214        -0.0036

=== LCX: B minus A, paired folds ===
 Fold  Delta balanced accuracy  Delta ROC-AUC
    1                  -0.0333        -0.0404
    2                  -0.0833        -0.0982
    3                   0.0526         0.0073
    4                  -0.0699        -0.0054
    5                  -0.0608        -0.0254

=== RCA: B minus A, paired folds ===
 Fold  Delta balanc

In [ ]:

# CardioVision3D — Step 5: Recall and specificity across CV folds

from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, recall_score
import numpy as np
import pandas as pd

def positive_recall(y_true, y_pred):
    positive = "CAD" if "CAD" in set(y_true) else "Stenotic"
    return recall_score(
        y_true, y_pred, pos_label=positive, zero_division=0
    )

def specificity(y_true, y_pred):
    positive = "CAD" if "CAD" in set(y_true) else "Stenotic"
    negative = "Normal"
    return recall_score(
        y_true, y_pred, pos_label=negative, zero_division=0
    )

diagnostic_rows = []

for target in ["Cath", "LAD", "LCX", "RCA"]:
    X_train_target, _, y_train_target, _ = target_splits[target]

    for set_name, columns in feature_sets.items():
        X = X_train_target[columns].copy()
        numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
        categorical_cols = [c for c in X.columns if c not in numeric_cols]

        prep = ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encode", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_cols),
        ])

        model = Pipeline([
            ("prep", prep),
            ("clf", LogisticRegression(max_iter=2000, random_state=42)),
        ])

        scores = cross_validate(
            model, X, y_train_target,
            cv=cv,
            scoring={
                "balanced_accuracy": "balanced_accuracy",
                "positive_recall": make_scorer(positive_recall),
                "specificity": make_scorer(specificity),
            },
            n_jobs=-1,
            error_score="raise",
        )

        diagnostic_rows.append({
            "Target": target,
            "Feature set": set_name,
            "Balanced accuracy": scores["test_balanced_accuracy"].mean(),
            "Positive recall": scores["test_positive_recall"].mean(),
            "Specificity": scores["test_specificity"].mean(),
        })

print(pd.DataFrame(diagnostic_rows).round(4).to_string(index=False))


Target   Feature set  Balanced accuracy  Positive recall  Specificity
  Cath    A_Basic_42             0.8155           0.9190       0.7121
  Cath B_Expanded_52             0.8485           0.9134       0.7835
   LAD    A_Basic_42             0.6673           0.7303       0.6043
   LAD B_Expanded_52             0.7093           0.7443       0.6743
   LCX    A_Basic_42             0.6243           0.4737       0.7749
   LCX B_Expanded_52             0.5854           0.4632       0.7076
   RCA    A_Basic_42             0.6487           0.5164       0.7811
   RCA B_Expanded_52             0.6609           0.5608       0.7611


In [ ]:

# CardioVision3D — Step 6: Training-only threshold analysis
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, balanced_accuracy_score

thresholds = [0.30, 0.40, 0.50, 0.60]
threshold_rows = []

for target in ["LCX", "RCA"]:
    X_train_target, _, y_train_target, _ = target_splits[target]
    positive_label = "Stenotic"
    y_binary = (y_train_target == positive_label).astype(int)

    for set_name, columns in feature_sets.items():
        X = X_train_target[columns].copy()
        numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
        categorical_cols = [c for c in X.columns if c not in numeric_cols]

        prep = ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]), numeric_cols),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encode", OneHotEncoder(handle_unknown="ignore")),
            ]), categorical_cols),
        ])

        model = Pipeline([
            ("prep", prep),
            ("clf", LogisticRegression(max_iter=2000, random_state=42)),
        ])

        # Each row receives a probability from a model that did not train on it.
        oof_prob = cross_val_predict(
            model, X, y_train_target,
            cv=cv, method="predict_proba", n_jobs=-1
        )

        classes = cross_val_predict(
            model, X, y_train_target,
            cv=cv, method="predict", n_jobs=-1
        )  # used only to keep the same CV configuration; probabilities below are authoritative

        # Cross-validated predictions use a consistent class order for this target.
        # Determine positive probability from the full training-label class order.
        class_order = sorted(pd.unique(y_train_target))
        positive_index = list(model.fit(X, y_train_target).classes_).index(positive_label)
        positive_prob = oof_prob[:, positive_index]

        for threshold in thresholds:
            pred_positive = (positive_prob >= threshold).astype(int)

            threshold_rows.append({
                "Target": target,
                "Feature set": set_name,
                "Threshold": threshold,
                "Recall": recall_score(
                    y_binary, pred_positive, zero_division=0
                ),
                "Specificity": recall_score(
                    1 - y_binary, 1 - pred_positive, zero_division=0
                ),
                "Balanced accuracy": balanced_accuracy_score(
                    y_binary, pred_positive
                ),
            })

threshold_df = pd.DataFrame(threshold_rows)
print(threshold_df.round(4).to_string(index=False))


Target   Feature set  Threshold  Recall  Specificity  Balanced accuracy
   LCX    A_Basic_42        0.3  0.7263       0.5782             0.6523
   LCX    A_Basic_42        0.4  0.6211       0.7007             0.6609
   LCX    A_Basic_42        0.5  0.4737       0.7755             0.6246
   LCX    A_Basic_42        0.6  0.3474       0.8299             0.5887
   LCX B_Expanded_52        0.3  0.6632       0.6190             0.6411
   LCX B_Expanded_52        0.4  0.6211       0.6667             0.6439
   LCX B_Expanded_52        0.5  0.4632       0.7075             0.5853
   LCX B_Expanded_52        0.6  0.4000       0.8027             0.6014
   RCA    A_Basic_42        0.3  0.6813       0.6291             0.6552
   RCA    A_Basic_42        0.4  0.6044       0.6954             0.6499
   RCA    A_Basic_42        0.5  0.5165       0.7815             0.6490
   RCA    A_Basic_42        0.6  0.4066       0.8477             0.6271
   RCA B_Expanded_52        0.3  0.7033       0.6490            

In [ ]:

# CardioVision3D — Step 7: Nested CV for LCX and RCA
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, recall_score

THRESHOLDS = [0.30, 0.40, 0.50, 0.60]
OUTER_FOLDS = 5
INNER_FOLDS = 4

def build_model(X):
    numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    prep = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ])

    return Pipeline([
        ("prep", prep),
        ("clf", LogisticRegression(max_iter=2000, random_state=42)),
    ])

nested_rows = []

for target in ["LCX", "RCA"]:
    X_all, _, y_all, _ = target_splits[target]
    y_all = np.asarray(y_all)
    positive_label = "Stenotic"

    outer_cv = StratifiedKFold(
        n_splits=OUTER_FOLDS, shuffle=True, random_state=123
    )

    for feature_name, columns in feature_sets.items():
        X = X_all[columns].copy()

        for outer_fold, (outer_train_idx, outer_valid_idx) in enumerate(
            outer_cv.split(X, y_all), start=1
        ):
            X_outer_train = X.iloc[outer_train_idx]
            X_outer_valid = X.iloc[outer_valid_idx]
            y_outer_train = y_all[outer_train_idx]
            y_outer_valid = y_all[outer_valid_idx]

            # Inner out-of-fold probabilities select the threshold.
            inner_cv = StratifiedKFold(
                n_splits=INNER_FOLDS, shuffle=True, random_state=42
            )
            inner_oof_prob = np.zeros(len(y_outer_train))

            for inner_train_idx, inner_valid_idx in inner_cv.split(
                X_outer_train, y_outer_train
            ):
                inner_model = build_model(
                    X_outer_train.iloc[inner_train_idx]
                )
                inner_model.fit(
                    X_outer_train.iloc[inner_train_idx],
                    y_outer_train[inner_train_idx]
                )

                positive_idx = list(inner_model.classes_).index(positive_label)
                inner_oof_prob[inner_valid_idx] = inner_model.predict_proba(
                    X_outer_train.iloc[inner_valid_idx]
                )[:, positive_idx]

            y_inner_binary = (y_outer_train == positive_label).astype(int)

            threshold_scores = []
            for threshold in THRESHOLDS:
                inner_pred = (inner_oof_prob >= threshold).astype(int)
                score = balanced_accuracy_score(
                    y_inner_binary, inner_pred
                )
                threshold_scores.append((score, threshold))

            # If scores tie, choose the threshold closest to 0.50.
            best_score = max(score for score, _ in threshold_scores)
            tied = [
                threshold for score, threshold in threshold_scores
                if np.isclose(score, best_score)
            ]
            chosen_threshold = min(tied, key=lambda t: abs(t - 0.50))

            # Refit on the outer training fold and evaluate once on outer validation.
            outer_model = build_model(X_outer_train)
            outer_model.fit(X_outer_train, y_outer_train)
            positive_idx = list(outer_model.classes_).index(positive_label)

            valid_prob = outer_model.predict_proba(X_outer_valid)[:, positive_idx]
            valid_pred = (valid_prob >= chosen_threshold).astype(int)
            y_valid_binary = (y_outer_valid == positive_label).astype(int)

            nested_rows.append({
                "Target": target,
                "Feature set": feature_name,
                "Outer fold": outer_fold,
                "Selected threshold": chosen_threshold,
                "Balanced accuracy": balanced_accuracy_score(
                    y_valid_binary, valid_pred
                ),
                "Recall": recall_score(
                    y_valid_binary, valid_pred, zero_division=0
                ),
                "Specificity": recall_score(
                    1 - y_valid_binary, 1 - valid_pred, zero_division=0
                ),
            })

nested_df = pd.DataFrame(nested_rows)

print("=== OUTER-FOLD RESULTS ===")
print(nested_df.round(4).to_string(index=False))

print("\n=== NESTED CV SUMMARY ===")
summary = nested_df.groupby(["Target", "Feature set"]).agg(
    balanced_accuracy_mean=("Balanced accuracy", "mean"),
    balanced_accuracy_std=("Balanced accuracy", "std"),
    recall_mean=("Recall", "mean"),
    specificity_mean=("Specificity", "mean"),
    selected_thresholds=("Selected threshold",
                         lambda x: list(x))
).reset_index()

print(summary.round(4).to_string(index=False))


=== OUTER-FOLD RESULTS ===
Target   Feature set  Outer fold  Selected threshold  Balanced accuracy  Recall  Specificity
   LCX    A_Basic_42           1                 0.3             0.7404  0.9474       0.5333
   LCX    A_Basic_42           2                 0.4             0.6395  0.5789       0.7000
   LCX    A_Basic_42           3                 0.3             0.6697  0.6842       0.6552
   LCX    A_Basic_42           4                 0.3             0.7740  0.7895       0.7586
   LCX    A_Basic_42           5                 0.5             0.5118  0.3684       0.6552
   LCX B_Expanded_52           1                 0.3             0.6088  0.6842       0.5333
   LCX B_Expanded_52           2                 0.4             0.5965  0.5263       0.6667
   LCX B_Expanded_52           3                 0.3             0.6525  0.6842       0.6207
   LCX B_Expanded_52           4                 0.4             0.6860  0.5789       0.7931
   LCX B_Expanded_52           5           

In [ ]:

required = [
    "df",
    "target_models",
    "target_splits",
    "basic_features",
    "expanded_features",
]

for name in required:
    print(f"{name}: {'available' if name in globals() else 'MISSING'}")


df: available
target_models: available
target_splits: available
basic_features: available
expanded_features: available


In [ ]:

import numpy as np

print("=" * 65)
print("CARDIOVISION3D — MODEL PACKAGING AUDIT")
print("=" * 65)

print("\n1. DATASET")
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()))

print("\n2. FEATURE SETS")
print("Basic features:", len(basic_features))
print("Expanded features:", len(expanded_features))
print("Combined expanded set:", len(set(basic_features + expanded_features)))

print("\n3. TARGET MODEL AUDIT")

targets = ["Cath", "LAD", "LCX", "RCA"]
for target in targets:
    print(f"\n--- {target} ---")

    if target not in target_models:
        print("ERROR: model missing")
        continue

    if target not in target_splits:
        print("ERROR: split missing")
        continue

    model = target_models[target]
    X_train, X_test, y_train, y_test = target_splits[target]

    print("Model type:", type(model).__name__)
    print("Training rows:", len(X_train))
    print("Test rows:", len(X_test))
    print("Training feature count:", X_train.shape[1])
    print("Test feature count:", X_test.shape[1])
    print("Classes:", list(model.classes_))
    print("Pipeline steps:", list(model.named_steps.keys()))

    print("Targets excluded:",
          all(t not in X_train.columns
              for t in ["Cath", "LAD", "LCX", "RCA"]))

    try:
        probabilities = model.predict_proba(X_test.iloc[:2])
        print("Probability output shape:", probabilities.shape)
        print("Probabilities finite:", bool(np.isfinite(probabilities).all()))
        print("Probability sums valid:",
              bool(np.allclose(probabilities.sum(axis=1), 1.0)))
    except Exception as exc:
        print("Probability check failed:", repr(exc))

print("\n" + "=" * 65)
print("AUDIT COMPLETE — review any errors before packaging")
print("=" * 65)


CARDIOVISION3D — MODEL PACKAGING AUDIT

1. DATASET
Shape: (303, 59)
Missing values: 0

2. FEATURE SETS
Basic features: 42
Expanded features: 10
Combined expanded set: 52

3. TARGET MODEL AUDIT

--- Cath ---
Model type: Pipeline
Training rows: 242
Test rows: 61
Training feature count: 52
Test feature count: 52
Classes: ['CAD', 'Normal']
Pipeline steps: ['preprocessor', 'classifier']
Targets excluded: True
Probability output shape: (2, 2)
Probabilities finite: True
Probability sums valid: True

--- LAD ---
Model type: Pipeline
Training rows: 242
Test rows: 61
Training feature count: 52
Test feature count: 52
Classes: ['Normal', 'Stenotic']
Pipeline steps: ['preprocessor', 'classifier']
Targets excluded: True
Probability output shape: (2, 2)
Probabilities finite: True
Probability sums valid: True

--- LCX ---
Model type: Pipeline
Training rows: 242
Test rows: 61
Training feature count: 52
Test feature count: 52
Classes: ['Normal', 'Stenotic']
Pipeline steps: ['preprocessor', 'classifier']

In [ ]:

# ============================================================
# CardioVision3D — FINAL MODEL PACKAGING
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ------------------------------------------------------------
# 1. Verify required notebook objects
# ------------------------------------------------------------

required_objects = [
    "df",
    "target_models",
    "target_splits",
    "basic_features",
    "expanded_features",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        f"Missing notebook objects: {missing_objects}. "
        "Restore the data/model preparation cells first."
    )

TARGETS = ["Cath", "LAD", "LCX", "RCA"]
TARGET_COLUMNS = set(TARGETS)

# Provisional feature choices based on earlier CV experiments.
selected_features = {
    "Cath": list(basic_features) + list(expanded_features),
    "LAD":  list(basic_features) + list(expanded_features),
    "LCX":  list(basic_features),
    "RCA":  list(basic_features) + list(expanded_features),
}

# Preserve order while removing accidental duplicates.
selected_features = {
    target: list(dict.fromkeys(features))
    for target, features in selected_features.items()
}

# ------------------------------------------------------------
# 2. Create output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path("cardiovision3d_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. Build fresh preprocessing + classifier pipeline
# ------------------------------------------------------------

def build_pipeline(X):
    numeric_columns = X.select_dtypes(
        include=["number"]
    ).columns.tolist()

    categorical_columns = [
        column for column in X.columns
        if column not in numeric_columns
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        ),
    ])

    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        (
            "categorical",
            categorical_pipeline,
            categorical_columns
        ),
    ])

    return Pipeline([
        ("preprocessor", preprocessing),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        ),
    ])

# ------------------------------------------------------------
# 4. Train selected models on training splits and save
# ------------------------------------------------------------

metadata = {
    "project": "CardioVision3D",
    "status": "Provisional research prototype",
    "dataset_shape": list(df.shape),
    "split_seed": 42,
    "training_note": (
        "Models fitted only on existing training splits. "
        "Held-out test data is not used for fitting or selection."
    ),
    "sklearn_version": sklearn.__version__,
    "models": {},
}

for target in TARGETS:
    print(f"\n{'=' * 55}")
    print(f"Processing target: {target}")
    print("=" * 55)

    if target not in target_splits:
        raise KeyError(f"Missing split for {target}")

    X_train, X_test, y_train, y_test = target_splits[target]
    features = selected_features[target]

    # Validate features and target-leakage protection.
    missing_features = [
        feature for feature in features
        if feature not in X_train.columns
    ]

    if missing_features:
        raise ValueError(
            f"{target}: missing features: {missing_features}"
        )

    if TARGET_COLUMNS.intersection(features):
        raise ValueError(
            f"{target}: target leakage detected in features"
        )

    if len(features) != (42 if target == "LCX" else 52):
        raise ValueError(
            f"{target}: unexpected feature count {len(features)}"
        )

    X_train_selected = X_train.loc[:, features]
    X_test_selected = X_test.loc[:, features]

    # Fit a fresh pipeline using only training data.
    model = build_pipeline(X_train_selected)
    model.fit(X_train_selected, y_train)

    # Verify prediction probabilities on held-out rows.
    probabilities = model.predict_proba(
        X_test_selected.iloc[:2]
    )

    assert probabilities.shape == (2, len(model.classes_))
    assert np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1), 1.0)

    # Save the complete fitted pipeline.
    model_filename = f"{target.lower()}_pipeline.joblib"
    model_path = OUTPUT_DIR / model_filename
    joblib.dump(model, model_path)

    # Save ordered input feature names.
    features_filename = f"{target.lower()}_features.json"
    features_path = OUTPUT_DIR / features_filename
    features_path.write_text(
        json.dumps(features, indent=2),
        encoding="utf-8"
    )

    positive_class = "CAD" if target == "Cath" else "Stenotic"

    metadata["models"][target] = {
        "artifact": model_filename,
        "features_file": features_filename,
        "feature_count": len(features),
        "classes": [str(value) for value in model.classes_],
        "positive_class": positive_class,
        "probability_threshold": None,
        "threshold_status": "Not finalized",
        "training_rows": int(len(X_train_selected)),
        "reserved_test_rows": int(len(X_test_selected)),
        "test_set_used_for_fitting": False,
    }

    print("Features:", len(features))
    print("Classes:", list(model.classes_))
    print("Training rows:", len(X_train_selected))
    print("Probability check: PASS")
    print("Saved:", model_path)

# ------------------------------------------------------------
# 5. Save metadata
# ------------------------------------------------------------

metadata_path = OUTPUT_DIR / "metadata.json"
metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

# ------------------------------------------------------------
# 6. Reload every saved pipeline and verify
# ------------------------------------------------------------

print(f"\n{'=' * 55}")
print("RELOAD VERIFICATION")
print("=" * 55)

for target in TARGETS:
    model_path = OUTPUT_DIR / (
        metadata["models"][target]["artifact"]
    )

    features = selected_features[target]
    X_train, X_test, y_train, y_test = target_splits[target]
    X_test_selected = X_test.loc[:, features]

    loaded_model = joblib.load(model_path)
    test_probabilities = loaded_model.predict_proba(
        X_test_selected.iloc[:2]
    )

    assert list(loaded_model.classes_) == (
        metadata["models"][target]["classes"]
    )
    assert test_probabilities.shape == (
        2, len(loaded_model.classes_)
    )
    assert np.isfinite(test_probabilities).all()

    print(f"{target}: PASS")

# ------------------------------------------------------------
# 7. Print artifact summary
# ------------------------------------------------------------

print(f"\n{'=' * 55}")
print("PACKAGING COMPLETED SUCCESSFULLY")
print("=" * 55)
print("Output directory:", OUTPUT_DIR.resolve())
print("Metadata file:", metadata_path.resolve())

print("\nSaved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {path.name} ({path.stat().st_size:,} bytes)")

print("\nFeature configuration:")
for target in TARGETS:
    print(
        f"  {target}: "
        f"{metadata['models'][target]['feature_count']} features"
    )

print("\nAll saved models passed reload verification.")
print(
    "Reminder: these are research prototype predictions, "
    "not clinically validated diagnostic results."
)



Processing target: Cath
Features: 52
Classes: ['CAD', 'Normal']
Training rows: 242
Probability check: PASS
Saved: cardiovision3d_artifacts/cath_pipeline.joblib

Processing target: LAD
Features: 52
Classes: ['Normal', 'Stenotic']
Training rows: 242
Probability check: PASS
Saved: cardiovision3d_artifacts/lad_pipeline.joblib

Processing target: LCX
Features: 42
Classes: ['Normal', 'Stenotic']
Training rows: 242
Probability check: PASS
Saved: cardiovision3d_artifacts/lcx_pipeline.joblib

Processing target: RCA
Features: 52
Classes: ['Normal', 'Stenotic']
Training rows: 242
Probability check: PASS
Saved: cardiovision3d_artifacts/rca_pipeline.joblib

RELOAD VERIFICATION
Cath: PASS
LAD: PASS
LCX: PASS
RCA: PASS

PACKAGING COMPLETED SUCCESSFULLY
Output directory: /content/cardiovision3d_artifacts
Metadata file: /content/cardiovision3d_artifacts/metadata.json

Saved files:
  cath_features.json (662 bytes)
  cath_pipeline.joblib (11,290 bytes)
  lad_features.json (662 bytes)
  lad_pipeline.jobl

In [ ]:

from pathlib import Path
import importlib.util
import json

# Use the directory created by the packaging cell.
ARTIFACT_DIR = Path("/content/cardiovision3d_artifacts")

assert (ARTIFACT_DIR / "metadata.json").exists(), (
    f"Artifacts not found at {ARTIFACT_DIR}"
)

inference_source = r'''
from pathlib import Path
from collections.abc import Mapping
import json
import joblib
import pandas as pd

ARTIFACT_DIR = Path(__file__).resolve().parent

with open(ARTIFACT_DIR / "metadata.json", encoding="utf-8") as f:
    METADATA = json.load(f)

TARGETS = ["Cath", "LAD", "LCX", "RCA"]
MODELS = {}
FEATURES = {}

# Load the exact saved pipelines and feature manifests.
for target in TARGETS:
    config = METADATA["models"][target]

    with open(
        ARTIFACT_DIR / config["features_file"],
        encoding="utf-8"
    ) as f:
        FEATURES[target] = json.load(f)

    MODELS[target] = joblib.load(
        ARTIFACT_DIR / config["artifact"]
    )

def predict_all(patient_features):
    """
    Input:
        Mapping from feature names to patient feature values.

    Output:
        Four model predictions and their class probabilities.

    This is a research prototype interface, not a diagnostic tool.
    """

    if not isinstance(patient_features, Mapping):
        raise TypeError(
            "patient_features must be a dictionary or mapping."
        )

    # Validate the complete input schema before predicting.
    required_features = set(
        feature
        for target in TARGETS
        for feature in FEATURES[target]
    )

    missing = sorted(required_features - set(patient_features))

    if missing:
        raise ValueError(
            "Missing required input features: "
            + ", ".join(missing)
        )

    outputs = {}

    for target in TARGETS:
        feature_names = FEATURES[target]

        # Preserve the exact training feature order.
        row = pd.DataFrame(
            [{
                feature: patient_features[feature]
                for feature in feature_names
            }],
            columns=feature_names
        )

        model = MODELS[target]
        probabilities = model.predict_proba(row)[0]
        predicted_class = str(model.predict(row)[0])
        classes = [str(c) for c in model.classes_]

        class_probabilities = {
            label: float(probability)
            for label, probability in zip(classes, probabilities)
        }

        positive_class = METADATA["models"][target]["positive_class"]

        outputs[target] = {
            "predicted_class": predicted_class,
            "class_probabilities": class_probabilities,
            "positive_class": positive_class,
            "positive_class_model_probability":
                class_probabilities[positive_class],
            "threshold": None,
            "threshold_status": "Not finalized",
        }

    return {
        "project": "CardioVision3D",
        "status": "research_prototype",
        "outputs": outputs,
        "warning": (
            "Research predictions only. Model probabilities are not "
            "clinically validated disease risks and must not be used "
            "as a standalone diagnosis."
        ),
    }
'''

# Save the module beside the model artifacts.
inference_path = ARTIFACT_DIR / "inference.py"
inference_path.write_text(inference_source, encoding="utf-8")

# Import the newly created module.
spec = importlib.util.spec_from_file_location(
    "cardiovision3d_inference",
    inference_path
)
inference_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference_module)

# Smoke test using one existing TRAINING row.
# This tests software integration, not model accuracy.
X_train = target_splits["Cath"][0]
sample_features = X_train.iloc[0].to_dict()

result = inference_module.predict_all(sample_features)

print("=" * 60)
print("CARDIOVISION3D INFERENCE SMOKE TEST")
print("=" * 60)

for target, output in result["outputs"].items():
    print(f"\n{target}")
    print("Predicted class:", output["predicted_class"])
    print("Class probabilities:", output["class_probabilities"])
    print(
        "Positive-class model probability:",
        round(output["positive_class_model_probability"], 4)
    )

assert list(result["outputs"].keys()) == [
    "Cath", "LAD", "LCX", "RCA"
]

assert all(
    len(output["class_probabilities"]) == 2
    for output in result["outputs"].values()
)

print("\nAll four outputs generated successfully.")
print("Inference module:", inference_path)
print("Smoke test passed.")
print("\nFiles now available:")
for path in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", path.name)


CARDIOVISION3D INFERENCE SMOKE TEST

Cath
Predicted class: CAD
Class probabilities: {'CAD': 0.9942342351055933, 'Normal': 0.005765764894406765}
Positive-class model probability: 0.9942

LAD
Predicted class: Stenotic
Class probabilities: {'Normal': 0.29042090721604175, 'Stenotic': 0.7095790927839583}
Positive-class model probability: 0.7096

LCX
Predicted class: Stenotic
Class probabilities: {'Normal': 0.48487623036451244, 'Stenotic': 0.5151237696354876}
Positive-class model probability: 0.5151

RCA
Predicted class: Stenotic
Class probabilities: {'Normal': 0.22780368218453406, 'Stenotic': 0.7721963178154659}
Positive-class model probability: 0.7722

All four outputs generated successfully.
Inference module: /content/cardiovision3d_artifacts/inference.py
Smoke test passed.

Files now available:
 - __pycache__
 - cath_features.json
 - cath_pipeline.joblib
 - inference.py
 - lad_features.json
 - lad_pipeline.joblib
 - lcx_features.json
 - lcx_pipeline.joblib
 - metadata.json
 - rca_feature

In [ ]:

# ============================================================
# CardioVision3D — INFERENCE CONTRACT TESTS
# ============================================================

import numpy as np
import json
import importlib.util
from pathlib import Path

ARTIFACT_DIR = Path("/content/cardiovision3d_artifacts")
INFERENCE_PATH = ARTIFACT_DIR / "inference.py"

spec = importlib.util.spec_from_file_location(
    "cardiovision3d_inference_tests",
    INFERENCE_PATH
)
inference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(inference)

# Use a training row strictly for software testing.
X_train = target_splits["Cath"][0]
sample = X_train.iloc[0].to_dict()

passed = 0
failed = 0

def check(name, test_function):
    global passed, failed
    try:
        test_function()
        passed += 1
        print(f"PASS: {name}")
    except Exception as exc:
        failed += 1
        print(f"FAIL: {name} -> {type(exc).__name__}: {exc}")

# Test 1: All four targets are returned.
def test_four_targets():
    result = inference.predict_all(sample)
    assert set(result["outputs"]) == {"Cath", "LAD", "LCX", "RCA"}

check("Returns all four targets", test_four_targets)

# Test 2: Every probability is finite and sums to one.
def test_probabilities():
    result = inference.predict_all(sample)
    for target, output in result["outputs"].items():
        probs = list(output["class_probabilities"].values())
        assert len(probs) == 2
        assert np.isfinite(probs).all()
        assert all(0 <= p <= 1 for p in probs)
        assert np.isclose(sum(probs), 1.0)
        assert output["predicted_class"] in (
            output["class_probabilities"]
        )

check("Valid probability distributions", test_probabilities)

# Test 3: Missing required fields are rejected.
def test_missing_fields():
    incomplete = dict(sample)
    incomplete.pop(next(iter(inference.FEATURES["Cath"])))
    try:
        inference.predict_all(incomplete)
    except ValueError as exc:
        assert "Missing required input features" in str(exc)
        return
    raise AssertionError("Missing input was not rejected")

check("Rejects incomplete input", test_missing_fields)

# Test 4: Extra fields do not break prediction.
def test_extra_fields():
    extended = dict(sample)
    extended["unused_test_field"] = "ignored"
    result = inference.predict_all(extended)
    assert len(result["outputs"]) == 4

check("Tolerates extra fields", test_extra_fields)

# Test 5: Unsupported input type is rejected.
def test_invalid_input_type():
    try:
        inference.predict_all(None)
    except TypeError:
        return
    raise AssertionError("Invalid input type was not rejected")

check("Rejects invalid input type", test_invalid_input_type)

print("\n" + "=" * 55)
print(f"RESULT: {passed} passed, {failed} failed")
print("=" * 55)

if failed:
    raise RuntimeError(
        "Resolve the failed tests before backend integration."
    )

print("Inference contract tests completed successfully.")


PASS: Returns all four targets
PASS: Valid probability distributions
PASS: Rejects incomplete input
PASS: Tolerates extra fields
PASS: Rejects invalid input type

RESULT: 5 passed, 0 failed
Inference contract tests completed successfully.


In [ ]:

from pathlib import Path
import zipfile
import json

ARTIFACT_DIR = Path("/content/cardiovision3d_artifacts")
ZIP_PATH = Path("/content/cardiovision3d_ml_package.zip")

required_files = [
    "cath_pipeline.joblib",
    "lad_pipeline.joblib",
    "lcx_pipeline.joblib",
    "rca_pipeline.joblib",
    "cath_features.json",
    "lad_features.json",
    "lcx_features.json",
    "rca_features.json",
    "metadata.json",
    "inference.py",
]

missing = [
    name for name in required_files
    if not (ARTIFACT_DIR / name).is_file()
]

if missing:
    raise FileNotFoundError(f"Missing artifacts: {missing}")

# Verify the metadata is readable before exporting.
metadata = json.loads(
    (ARTIFACT_DIR / "metadata.json").read_text(encoding="utf-8")
)
assert set(metadata["models"]) == {"Cath", "LAD", "LCX", "RCA"}

with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in required_files:
        archive.write(ARTIFACT_DIR / name, arcname=name)

print("ML package created:", ZIP_PATH)
print("Size:", f"{ZIP_PATH.stat().st_size:,} bytes")
print("\nContents:")
with zipfile.ZipFile(ZIP_PATH) as archive:
    for name in archive.namelist():
        print(" -", name)

# In Google Colab, trigger a download to your computer.
from google.colab import files
files.download(str(ZIP_PATH))


ML package created: /content/cardiovision3d_ml_package.zip
Size: 18,822 bytes

Contents:
 - cath_pipeline.joblib
 - lad_pipeline.joblib
 - lcx_pipeline.joblib
 - rca_pipeline.joblib
 - cath_features.json
 - lad_features.json
 - lcx_features.json
 - rca_features.json
 - metadata.json
 - inference.py


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>